In [1]:
import os
import socket
import requests
import urllib3.util.connection as urllib3_cn
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
API_KEY = os.getenv("DATA_GOV_API_KEY")
RESOURCE_ID = "9ef84268-d588-465a-a308-a864a43d0070"

# 1. Force IPv4 (prevents 30s hangs on broken IPv6 routes)
def allowed_gai_family():
    return socket.AF_INET

urllib3_cn.allowed_gai_family = allowed_gai_family

# 2. Create a session with browser headers & automatic retries
session = requests.Session()
retries = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json",
}

ENDPOINTS = [
    f"https://api.data.gov.in/resource/{RESOURCE_ID}",
    f"https://www.data.gov.in/api/datastore/resource.json?resource_id={RESOURCE_ID}",
]

params = {
    "api-key": API_KEY,
    "format": "json",
    "offset": 0,
    "limit": 5,
}

for url in ENDPOINTS:
    try:
        print(f"Testing: {url} ...")
        # (10s connect timeout, 60s read timeout)
        resp = session.get(url, params=params, headers=HEADERS, timeout=(10, 60))
        print("Status Code:", resp.status_code)
        data = resp.json()
        print("Success! Sample keys:", list(data.get("records", [{}])[0].keys()))
        WORKING_URL = url
        break
    except Exception as e:
        print(f"Failed on {url} -> {type(e).__name__}: {e}\n")

Testing: https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070 ...
Status Code: 200
Success! Sample keys: ['state', 'district', 'market', 'commodity', 'variety', 'grade', 'arrival_date', 'min_price', 'max_price', 'modal_price']


In [2]:
all_ap_records = []
offset = 0
limit = 100

while True:
    query = {
        "api-key": API_KEY,
        "format": "json",
        "offset": offset,
        "limit": limit,
        "filters[state]": "Andhra Pradesh",
    }
    r = session.get(WORKING_URL, params=query, headers=HEADERS, timeout=(10, 60)).json()
    batch = r.get("records", [])
    if not batch:
        break
    all_ap_records.extend(batch)
    offset += limit
    if offset >= int(r.get("total", 0)):
        break

df_ap = pd.DataFrame(all_ap_records)
df_ap.columns = [c.lower() for c in df_ap.columns]

if not df_ap.empty:
    df_ap["arrival_date"] = pd.to_datetime(df_ap["arrival_date"], dayfirst=True, errors="coerce")
    print(f"Total AP rows fetched: {len(df_ap)}")
    print(f"Date range in API: {df_ap['arrival_date'].min().date()} -> {df_ap['arrival_date'].max().date()}")
    print("\nTop 10 AP Markets:")
    print(df_ap["market"].value_counts().head(10))
    print("\nTop 10 Commodities in AP:")
    print(df_ap["commodity"].value_counts().head(10))
else:
    print("0 records returned for Andhra Pradesh.")

Total AP rows fetched: 3323
Date range in API: 2026-11-13 -> 2026-11-13

Top 10 AP Markets:
market
PMY Una             38
PMY Hamirpur        34
SMY Palampur        34
PMY Kullu           33
Katni APMC          31
PMY Kather Solan    30
Neemuch APMC        30
Mandsaur APMC       30
SMY Bhuntar         30
PMY Kangni Mandi    27
Name: count, dtype: int64

Top 10 Commodities in AP:
commodity
Paddy(Common)                   329
Wheat                           327
Maize                           298
Soyabean                        198
Bengal Gram(Gram)(Whole)        140
Black Gram(Urd Beans)(Whole)    101
Onion                            82
Potato                           78
Tomato                           73
Mustard                          66
Name: count, dtype: int64


In [3]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_params = {
    "latitude": 17.0005,
    "longitude": 81.8040,
    "start_date": "2026-10-01",
    "end_date": "2026-11-10",
    "daily": ["temperature_2m_max", "precipitation_sum", "rain_sum"],
    "timezone": "Asia/Kolkata",
}

w_resp = session.get(weather_url, params=weather_params, timeout=30).json()
df_weather = pd.DataFrame(w_resp["daily"])
print("Weather rows fetched:", len(df_weather))
display(df_weather.tail())

Weather rows fetched: 41


,time,temperature_2m_max,precipitation_sum,rain_sum
36,2026-11-06,31.0,0.5,0.5
37,2026-11-07,31.1,1.0,1.0
38,2026-11-08,31.1,1.5,1.5
39,2026-11-09,30.6,0.0,0.0
40,2026-11-10,30.9,0.0,0.0


In [4]:
import os
from dotenv import load_dotenv
import pandas as pd
import requests

load_dotenv()
CEDA_API_KEY = os.getenv("CEDA_API_KEY", "").strip().strip('"').strip("'")
CEDA_BASE_URL = "https://api.ceda.ashoka.edu.in/v1/agmarknet"

# Pass your CEDA key in headers (update header key if your original notebook used 'x-api-key' or 'api-key')
HEADERS = {
    "Authorization": f"Bearer {CEDA_API_KEY}",
    "x-api-key": CEDA_API_KEY,
    "Accept": "application/json",
}


def fetch_ceda_df(endpoint):
  resp = requests.get(f"{CEDA_BASE_URL}/{endpoint}", headers=HEADERS, timeout=20)
  if resp.status_code != 200:
    print(f"[{endpoint}] Error {resp.status_code}: {resp.text}")
    return pd.DataFrame()
  payload = resp.json()
  if isinstance(payload, list):
    return pd.DataFrame(payload)
  if isinstance(payload.get("output"), dict):
    return pd.DataFrame(payload["output"].get("data", []))
  return pd.DataFrame(payload.get("data", payload.get("records", [])))


comm_df = fetch_ceda_df("commodities")
if not comm_df.empty:
  print("Sample Commodities:")
  display(
      comm_df[
          comm_df.apply(
              lambda r: any(
                  k in str(r.values).lower()
                  for k in ["onion", "tomato", "chilli"]
              ),
              axis=1,
          )
      ]
  )

geo_df = fetch_ceda_df("geographies")
if not geo_df.empty:
  print("\nTarget States / Districts:")
  display(
      geo_df[
          geo_df.apply(
              lambda r: any(
                  k in str(r.values).lower()
                  for k in ["andhra", "nashik", "kolar"]
              ),
              axis=1,
          )
      ].head(15)
  )

Sample Commodities:


,commodity_id,commodity_name
22,23,Onion
64,78,Tomato
72,87,Green Chilli
113,132,Dry Chillies
306,358,Onion Green
421,491,Bajji chilli



Target States / Districts:


,census_state_id,census_state_name,census_district_id,census_district_name
515,27,Maharashtra,516,Nashik
541,28,Andhra Pradesh,542,Srikakulam
542,28,Andhra Pradesh,543,Vizianagaram
543,28,Andhra Pradesh,544,Visakhapatnam
544,28,Andhra Pradesh,545,East Godavari
545,28,Andhra Pradesh,546,West Godavari
546,28,Andhra Pradesh,547,Krishna
547,28,Andhra Pradesh,548,Guntur
548,28,Andhra Pradesh,549,Prakasam
549,28,Andhra Pradesh,550,Sri Potti Sriramulu Nellore


In [7]:
import pandas as pd
import requests

HEADERS = {
    "Authorization": f"Bearer {CEDA_API_KEY}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}


def fetch_ceda_markets(commodity_id, state_id, district_id=None):
  """Fetches market_id -> market_name mapping from CEDA /markets."""
  for d_val in ([district_id] if district_id else [None, 0]):
    body = {"commodity_id": commodity_id, "state_id": state_id}
    if d_val is not None:
      body["district_id"] = d_val
    resp = requests.post(
        f"{CEDA_BASE_URL}/markets", json=body, headers=HEADERS, timeout=25
    )
    if resp.status_code == 200:
      rows = resp.json().get("output", {}).get("data", [])
      if rows:
        return {
            int(r.get("market_id") or r.get("census_market_id")): (
                r.get("market_name")
                or r.get("market")
                or f"Market_{r.get('market_id')}"
            )
            for r in rows
            if (r.get("market_id") or r.get("census_market_id")) is not None
        }
  return {}


def fetch_ceda_history(commodity_id, state_id, district_ids=None, windows=None):
  if windows is None:
    windows = [
        ("2025-01-01", "2025-04-30"),
        ("2025-05-01", "2025-08-31"),
        ("2025-09-01", "2025-12-31"),
        ("2026-01-01", "2026-04-30"),
        ("2026-05-01", "2026-08-31"),
        ("2026-09-01", "2026-11-13"),
    ]

  all_rows = []
  for start_dt, end_dt in windows:
    payload = {
        "commodity_id": commodity_id,
        "state_id": state_id,
        "from_date": start_dt,
        "to_date": end_dt,
    }
    if district_ids:
      payload["district_id"] = district_ids

    resp = requests.post(
        f"{CEDA_BASE_URL}/prices",
        json=payload,
        headers=HEADERS,
        timeout=(10, 90),
    )
    if resp.status_code == 200:
      chunk = resp.json().get("output", {}).get("data", [])
      print(f"[{start_dt} -> {end_dt}] Fetched {len(chunk)} rows")
      all_rows.extend(chunk)
    else:
      print(f"[{start_dt} -> {end_dt}] HTTP {resp.status_code}")

  df = pd.DataFrame(all_rows)
  if df.empty:
    return df

  # Clean ISO timestamp to YYYY-MM-DD date
  df["date"] = pd.to_datetime(df["date"]).dt.date

  # Map market names
  dist_single = district_ids[0] if district_ids else None
  mkt_map = fetch_ceda_markets(commodity_id, state_id, dist_single)
  df["market_name"] = df["market_id"].map(
      lambda x: mkt_map.get(int(x), f"Mandi_{x}")
  )

  # Merge readable commodity and geography names from Cell 1 DataFrames
  if "comm_df" in globals() and not comm_df.empty:
    df = df.merge(
        comm_df[["commodity_id", "commodity_name"]],
        on="commodity_id",
        how="left",
    )
  if "geo_df" in globals() and not geo_df.empty:
    df = df.merge(
        geo_df[[
            "census_state_id",
            "census_state_name",
            "census_district_id",
            "census_district_name",
        ]].drop_duplicates(),
        on=["census_state_id", "census_district_id"],
        how="left",
    )

  return df


# Example: Fetch Nashik Onion (commodity_id=23, state_id=27, district_id=[516])
nashik_onion_df = fetch_ceda_history(
    commodity_id=23, state_id=27, district_ids=[516]
)
print(
    f"\nTotal rows: {len(nashik_onion_df)} | Date range:"
    f" {nashik_onion_df['date'].min()} to {nashik_onion_df['date'].max()}"
)
display(nashik_onion_df.head())

[2025-01-01 -> 2025-04-30] Fetched 1639 rows
[2025-05-01 -> 2025-08-31] HTTP 504
[2025-09-01 -> 2025-12-31] HTTP 504
[2026-01-01 -> 2026-04-30] Fetched 0 rows
[2026-05-01 -> 2026-08-31] Fetched 0 rows
[2026-09-01 -> 2026-11-13] Fetched 0 rows

Total rows: 1639 | Date range: 2025-01-01 to 2025-04-30


,date,commodity_id,census_state_id,census_district_id,market_id,min_price,max_price,modal_price,market_name,commodity_name,census_state_name,census_district_name
0,2025-01-01,23,27,516,170,951.0,3005,2300.0,Mandi_170,Onion,Maharashtra,Nashik
1,2025-01-01,23,27,516,10121,2100.0,2900,2660.0,Mandi_10121,Onion,Maharashtra,Nashik
2,2025-01-01,23,27,516,168,1100.0,2701,2300.0,Mandi_168,Onion,Maharashtra,Nashik
3,2025-01-01,23,27,516,169,1100.0,3019,2451.0,Mandi_169,Onion,Maharashtra,Nashik
4,2025-01-01,23,27,516,1815,500.0,2790,2400.0,Mandi_1815,Onion,Maharashtra,Nashik


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/food_price_intelligence.db")

prices_query = """
SELECT 
    dp.date,
    m.state,
    m.district,
    m.name AS market_name,
    c.name AS commodity,
    dp.min_price_inr,
    dp.max_price_inr,
    dp.modal_price_inr
FROM daily_prices dp
JOIN markets m ON dp.market_id = m.id
JOIN commodities c ON dp.commodity_id = c.id
ORDER BY dp.date DESC
"""

df_prices = pd.read_sql_query(prices_query, conn, parse_dates=["date"])
df_weather = pd.read_sql_query(
    "SELECT * FROM weather_data ORDER BY date DESC", conn, parse_dates=["date"]
)
conn.close()

print(f"Total Price Records in DB: {len(df_prices):,}")
print(f"Date Range: {df_prices['date'].min().date()} to {df_prices['date'].max().date()}")
print("\nRecords by State & Commodity:")
display(
    df_prices.groupby(["state", "commodity"])
    .agg(
        records=("modal_price_inr", "count"),
        min_date=("date", "min"),
        max_date=("date", "max"),
    )
    .reset_index()
)